# Classical Baselines: Logistic Regression and XGBoost for HIGGS Classification

**Author:** Aditya Rana (classical ML lead) | **Project:** Higgs Event Lab | **Module:** MID-03

This notebook trains and evaluates Logistic Regression and XGBoost across all three feature groups using the complete approximately 11-million-row UCI HIGGS dataset. Results are exported as `higgs-experiment-v1` artifacts for the Streamlit dashboard.

> **Scientific boundary:** All events are Monte Carlo simulations. Results are classification scores on synthetic benchmark events, not physical discovery claims.

---
**Run order:** This is notebook **3 of 3** in the mid-semester sequence.
1. `01_eda_and_feature_groups.ipynb` — full-data EDA
2. `02_neural_network_baseline.ipynb` — full-data MLP
3. **This notebook** — full-data Logistic Regression + XGBoost baselines

**Prerequisites:** `data/raw/HIGGS.csv.gz` must exist. The download cell obtains the complete source file.


## Step 0 — Complete-data setup

This cell ensures the complete raw HIGGS file exists. If it is missing, the official UCI archive is downloaded and extracted. All six model runs below read the complete raw file directly.


In [ ]:
from pathlib import Path
import sys
import urllib.request
import zipfile
import shutil

root = Path.cwd()
while not (root / "src" / "higgs_event_lab").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

raw_path = root / "data" / "raw" / "HIGGS.csv.gz"
processed_path = raw_path
full_row_count = 11_000_000
output_dir = root / "outputs"
raw_path.parent.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

URL = "https://archive.ics.uci.edu/static/public/280/higgs.zip"

if not raw_path.exists():
    print("Downloading the complete HIGGS dataset from UCI...")
    zip_path = raw_path.parent / "higgs_download.zip"
    urllib.request.urlretrieve(URL, zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        gz_name = [name for name in archive.namelist() if name.endswith(".csv.gz")][0]
        with archive.open(gz_name) as src, open(raw_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
    zip_path.unlink()

print("Database source:", URL)
print("Complete raw data ready:", raw_path)
print("Expected complete rows:", f"{full_row_count:,}")


## Step 1 — Environment and Package Verification

This cell prints package versions and confirms `xgboost` is available. XGBoost is required for the tree ensemble baseline. If it is missing, install it with `pip install xgboost`.

In [ ]:
import importlib
import platform

packages = ["numpy", "pandas", "sklearn", "xgboost", "matplotlib"]
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:15s} {mod.__version__}")
    except ModuleNotFoundError:
        print(f"{pkg:15s} NOT INSTALLED — run: pip install {pkg}")

print(f"Python: {platform.python_version()}, Platform: {platform.system()}")

## Step 2 — Hyperparameter Configuration

All hyperparameters are declared here so the notebook is fully reproducible. These settings match the documented mid-semester benchmark. The training seed is fixed at 42 to ensure consistent stratified splits across all feature groups and models.

**Logistic Regression:** Uses L-BFGS with `C=1.0` (inverse regularisation). A `StandardScaler` is embedded inside an `sklearn.Pipeline` to prevent data leakage — scaling statistics are computed only on training data.

**XGBoost:** Uses `max_depth=6`, `n_estimators=150`, and `learning_rate=0.1` for a fast but competitive mid-semester baseline. A stronger configuration (depth=7, 600 trees) is used in `notebooks/03_full/`.

In [ ]:
FEATURE_GROUPS = ("low_level", "high_level", "all")
N_ROWS = full_row_count
DATA_PATH = raw_path

print("Logistic Regression config:", LR_CONFIG)
print("XGBoost config:", XGB_CONFIG)


## Step 3 — Complete-data loading and split verification

We load the complete raw file using the project's validated chunked loader and create the canonical `higgs-split-v1` stratified partition. The exact full-data split is recorded in each output artifact.


In [ ]:
from higgs_event_lab.data import load_higgs_sample
from higgs_event_lab.protocol import SplitConfig, create_stratified_splits
import pandas as pd
import numpy as np

batch = load_higgs_sample(DATA_PATH, n_rows=N_ROWS, feature_group="all")
split = create_stratified_splits(batch.y.to_numpy(), SplitConfig(seed=42))

print(f"Dataset: {len(batch.X):,} rows loaded from the complete raw file")
print(f"Features: {batch.X.shape[1]} columns")
print(f"Signal fraction: {batch.y.mean():.4f}")
print(f"Split sizes — Train: {len(split.train):,}  Val: {len(split.validation):,}  Test: {len(split.test):,}")

y = batch.y.to_numpy()
for name, idx in [("Train", split.train), ("Val", split.validation), ("Test", split.test)]:
    print(f"  {name} signal fraction: {y[idx].mean():.4f}")
del batch, split, y


## Step 4 — Logistic Regression Across Feature Groups

We train one Logistic Regression model per feature group. A scikit-learn `Pipeline` wraps `StandardScaler` and `LogisticRegression` to ensure the scaler is fit only on training data — preventing information leakage into validation and test partitions.

The operating threshold is selected on the validation partition at a target signal efficiency of 50%, then applied frozen to the held-out test set. This mirrors the practice of selecting operating points before seeing test data in real analyses.

In [ ]:
from higgs_event_lab.classical import run_feature_group_benchmarks

print("Training Logistic Regression across feature groups...")
lr_results = run_feature_group_benchmarks(
    data_path=DATA_PATH,
    row_count=N_ROWS,
    output_base_dir=output_dir,
    config=LR_CONFIG,
    model_type="logreg",
    groups=FEATURE_GROUPS,
)

print("\nLogistic Regression Results:")
print(f"{'Run Key':<25} {'Val AUC':>9} {'Test AUC':>9} {'Brier':>7} {'Time(s)':>8}")
print("-" * 62)
for key, res in lr_results.items():
    val_auc = res["validation"]["metrics"]["roc_auc"]
    test_auc = res["test"]["metrics"]["roc_auc"]
    brier = res["test"]["metrics"]["brier_score"]
    t = res["training_seconds"]
    print(f"{key:<25} {val_auc:>9.4f} {test_auc:>9.4f} {brier:>7.4f} {t:>8.2f}s")

## Step 5 — XGBoost Across Feature Groups

XGBoost uses gradient-boosted decision trees which naturally handle feature interactions and do not require normalisation. The same stratified split as Logistic Regression is used to ensure a fair comparison — only the model and its scale of operation change.

Validation data is passed as an evaluation set to enable early stopping on the XGBoost side (though here we run the full `n_estimators` for a reproducible midsem baseline). The operating threshold is again selected on the validation partition.

In [ ]:
print("Training XGBoost across feature groups...")
xgb_results = run_feature_group_benchmarks(
    data_path=DATA_PATH,
    row_count=N_ROWS,
    output_base_dir=output_dir,
    xgb_config=XGB_CONFIG,
    model_type="xgboost",
    groups=FEATURE_GROUPS,
)

print("\nXGBoost Results:")
print(f"{'Run Key':<28} {'Val AUC':>9} {'Test AUC':>9} {'Brier':>7} {'Time(s)':>8}")
print("-" * 65)
for key, res in xgb_results.items():
    val_auc = res["validation"]["metrics"]["roc_auc"]
    test_auc = res["test"]["metrics"]["roc_auc"]
    brier = res["test"]["metrics"]["brier_score"]
    t = res["training_seconds"]
    print(f"{key:<28} {val_auc:>9.4f} {test_auc:>9.4f} {brier:>7.4f} {t:>8.2f}s")

## Step 6 — ROC Curve Comparison

We plot the ROC curves for all six runs (2 models × 3 feature groups) on a single figure. The diagonal dashed line represents a random classifier (AUC = 0.5). Higher curves indicate better discrimination. This figure directly addresses the research question: *do high-level invariant mass features add measurable discriminating power for classical models?*

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="#0a0e14")

GROUP_COLORS = {"low_level": "#00e5ff", "high_level": "#ffb300", "all": "#00ff9c"}
STYLE = dict(facecolor="#0a0e14", edgecolor="#1e2833")

for ax, (title, results) in zip(axes, [("Logistic Regression", lr_results), ("XGBoost", xgb_results)]):
    ax.set_facecolor("#111820")
    ax.plot([0, 1], [0, 1], color="#3d4550", linestyle="--", linewidth=1, label="Random (AUC=0.50)")
    for key, res in results.items():
        group = key.split("_", 1)[1]
        y_test = res["test"]["labels"]
        scores = res["test"]["probabilities"]
        auc = res["test"]["metrics"]["roc_auc"]
        fpr, tpr, _ = roc_curve(y_test, scores)
        ax.plot(fpr, tpr, color=GROUP_COLORS.get(group, "#ffffff"), linewidth=2,
                label=f"{group} (AUC={auc:.4f})")
    ax.set_xlabel("False Positive Rate", color="#8b949e")
    ax.set_ylabel("True Positive Rate", color="#8b949e")
    ax.set_title(title, color="#e6edf3", fontsize=12)
    ax.tick_params(colors="#8b949e")
    ax.legend(facecolor="#1e2833", edgecolor="#3d4550", labelcolor="#c9d1d9", fontsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor("#3d4550")

fig.suptitle("ROC Curves: Classical Baselines — Feature Group Comparison (11M events)",
             color="#e6edf3", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.savefig(output_dir / "roc_classical_comparison.png", dpi=150, bbox_inches="tight",
            facecolor="#0a0e14")
plt.show()
print("ROC curve saved to outputs/roc_classical_comparison.png")

## Step 7 — Summary table and interpretation

This table aggregates all six complete-data runs for comparison and direct inclusion in the project report. It highlights the measured model and feature-group combination with the best held-out ROC-AUC.


In [ ]:
import pandas as pd

rows = []
for model_label, results in [("Logistic Regression", lr_results), ("XGBoost", xgb_results)]:
    for key, res in results.items():
        group = key.split("_", 1)[1]
        m = res["test"]["metrics"]
        op = res["operating_point"]
        rows.append({
            "Model": model_label,
            "Feature Group": group,
            "# Features": 21 if group == "low_level" else (7 if group == "high_level" else 28),
            "Test ROC-AUC": round(m["roc_auc"], 4),
            "Test PR-AUC": round(m["pr_auc"], 4),
            "Brier Score": round(m["brier_score"], 4),
            "Test Accuracy": round(m.get("accuracy", float("nan")), 4),
            "Bkg Rejection @50%": round(op["test_background_rejection"], 4),
            "Train Time (s)": round(res["training_seconds"], 2),
        })

df = pd.DataFrame(rows).sort_values("Test ROC-AUC", ascending=False)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
print(df.to_string(index=False))

best = df.iloc[0]
print(f"\nBest classical baseline: {best['Model']} on {best['Feature Group']} features")
print(f"  Test ROC-AUC: {best['Test ROC-AUC']:.4f}")
print(f"  Background rejection @ 50% signal efficiency: {best['Bkg Rejection @50%']:.4f}")

## Step 8 — Verify exported full-data artifacts

All six complete-data runs have been exported to `outputs/` as `higgs-experiment-v1` artifacts. This final cell verifies that each run folder contains the required metadata and held-out predictions.


In [ ]:
expected_run_ids = [
    "logreg_low_level", "logreg_high_level", "logreg_all",
    "xgboost_low_level", "xgboost_high_level", "xgboost_all",
]

print("Artifact verification:")
all_ok = True
for run_id in expected_run_ids:
    run_dir = output_dir / run_id
    metrics_ok = (run_dir / "metrics.json").is_file()
    preds_ok = (run_dir / "test_predictions.csv").is_file()
    status = "OK" if (metrics_ok and preds_ok) else "MISSING"
    if not (metrics_ok and preds_ok):
        all_ok = False
    print(f"  {run_id:<28} metrics.json={'OK' if metrics_ok else 'MISSING':7}  "
          f"test_predictions.csv={'OK' if preds_ok else 'MISSING'}  [{status}]")

print()
if all_ok:
    print("All 6 classical baseline artifacts are valid. Launch the dashboard with:")
    print("  streamlit run streamlit_app.py")
else:
    print("Some artifacts are missing — re-run Steps 4 and 5.")